In [1]:
import numpy as np
from pathlib import Path
from astropy.io import fits

import importlib
import synthetic_fields

importlib.reload(synthetic_fields)

from synthetic_fields import generate_cube, save_cube

Load field_config

In [2]:
from field_config import config

print(config)

{'field_synth': 'velocity', 'model_gs': 'stable', 'alpha': 1.5, 'shape': (256, 256, 256), 'spacing': (1.0, 1.0, 1.0), 'len_scale': (16.0, 16.0, 16.0), 'angles': (0.0, 0.0, 0.0), 'seed': 19880127}


In [3]:
#test_config = config.copy()
#test_config["shape"] = (64, 64, 64)

Create cube

In [4]:
from synthetic_fields import generate_cube

field_3d, grid, model, m3d = generate_cube(config)

print(model)
print(f"M3D:    {m3d}")
print(f"Shape:  {field_3d.shape}")
print(f"Finite: {np.isfinite(field_3d).all()}")
print(f"Min:    {field_3d.min():.6f}")
print(f"Max:    {field_3d.max():.6f}")
print(f"Mean:   {field_3d.mean():.6f}")
print(f"Std:    {field_3d.std():.6f}")

Stable(dim=3, var=1.0, len_scale=16.0, alpha=1.5)
M3D:    1.5
Shape:  (64, 64, 64)
Finite: True
Min:    -3.781146
Max:    3.560000
Mean:   0.144235
Std:    1.004966


In [6]:
fits_path = save_cube(
    field_3d,
    config,
    m3d,
    output_dir="cubes",
    overwrite = True,
)

print(f"Saved: {fits_path}")

Saved: cubes/stable_3d_velocity_N64x64x64_seed19880127.fits


Load and verify export

In [7]:
from astropy.io import fits

with fits.open(fits_path) as hdul:
    saved_cube = hdul[0].data
    header = hdul[0].header

    print("Shape:", saved_cube.shape)
    print("Model:", header["MODEL"])
    print("Field:", header["FIELD"])
    print("M3D:", header["M3D"])

    print(
        "Values match:",
        np.array_equal(saved_cube, field_3d.astype(np.float32)),
    )

Shape: (64, 64, 64)
Model: stable
Field: velocity
M3D: 1.5
Values match: True


In [8]:
header

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -32 / array data type                                
NAXIS   =                    3 / number of array dimensions                     
NAXIS1  =                   64                                                  
NAXIS2  =                   64                                                  
NAXIS3  =                   64                                                  
MODEL   = 'stable  '           / GSTools covariance model                       
FIELD   = 'velocity'           / Intended role of the base scalar field         
SEED    =             19880127 / Random realization seed                        
M3D     =                  1.5 / Effective covariance shape exponent            
ALPHA   =                  1.5 / Stable alpha or fixed model exponent           
LSCALE_X=                 16.0                                                  
ANGLE_X =                  0